# Google Gemini API

**Module:** 08-llm-apis

**Notebook:** `06-google-gemini-api.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Overview** with clear contracts and failure modes
- Explain and apply **Text Generation** with clear contracts and failure modes
- Explain and apply **Multimodal** with clear contracts and failure modes
- Explain and apply **Embeddings** with clear contracts and failure modes
- Explain and apply **Safety Settings** with clear contracts and failure modes
- Explain and apply **System Instructions & Tools** with clear contracts and failure modes
- Explain and apply **AI Studio vs Vertex** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Google Gemini API

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Overview**
2. **Text Generation**
3. **Multimodal**
4. **Embeddings**
5. **Safety Settings**
6. **System Instructions & Tools**
7. **AI Studio vs Vertex**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Overview

### Definition
**Overview** is a core building block in 06-google-gemini-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Overview typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Overview: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Overview as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Overview as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Overview
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Overview when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Overview improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Overview" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Overview"
    notebook: str = "06-google-gemini-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Overview"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Overview"}
strong = {"definition": "Overview", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Overview"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Overview", "passed": len(checks)-len(failed), "failed": failed})


In [ ]:
# Demo: decision table for applying "Overview"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_overview", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


## Text Generation

### Definition
**Text Generation** is a core building block in 06-google-gemini-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Text Generation typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Text Generation: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Text Generation as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Text Generation as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Text Generation
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Text Generation when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Text Generation" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Text Generation"
    notebook: str = "06-google-gemini-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Text Generation"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Text Generation"}
strong = {"definition": "Text Generation", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Text Generation"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Text Generation", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Text Generation

**Situation:** A team wants to productionize a feature involving **Text Generation**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Multimodal

### Definition
**Multimodal** is a core building block in 06-google-gemini-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Multimodal typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Multimodal: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Multimodal as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Multimodal as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Multimodal
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Multimodal when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Multimodal" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Multimodal"
    notebook: str = "06-google-gemini-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Multimodal"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Multimodal"}
strong = {"definition": "Multimodal", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Multimodal"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Multimodal", "passed": len(checks)-len(failed), "failed": failed})


## Embeddings

### Definition
**Embeddings** is a core building block in 06-google-gemini-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Embeddings typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Embeddings: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Embeddings as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Embeddings as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Embeddings
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Embeddings when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Embeddings" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Embeddings"
    notebook: str = "06-google-gemini-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
def approx_tokens(text: str) -> int:
    return max(1, len(text) // 4)

def usage_account(prompt: str, completion: str, price_in=0.15, price_out=0.60):
    # prices are illustrative $/1M tokens
    tin, tout = approx_tokens(prompt), approx_tokens(completion)
    cost = (tin * price_in + tout * price_out) / 1_000_000
    return {"prompt_tokens": tin, "completion_tokens": tout, "usd_estimate": round(cost, 6)}

print(usage_account("system+user..." * 50, "answer..." * 20))


In [ ]:
# Streaming chunk assembler (shape similar to provider events)
chunks = [{"delta": "Hello"}, {"delta": ", "}, {"delta": "world"}]
out = []
for ch in chunks:
    out.append(ch["delta"])
    print("partial:", "".join(out))
print("final:", "".join(out))


### Worked scenario — Embeddings

**Situation:** A team wants to productionize a feature involving **Embeddings**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Safety Settings

### Definition
**Safety Settings** is a core building block in 06-google-gemini-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Safety Settings typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Safety Settings: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Safety Settings as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Safety Settings as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Safety Settings
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Safety Settings when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Safety Settings" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Safety Settings"
    notebook: str = "06-google-gemini-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Safety Settings"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Safety Settings"}
strong = {"definition": "Safety Settings", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Safety Settings"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Safety Settings", "passed": len(checks)-len(failed), "failed": failed})


## System Instructions & Tools

### Definition
**System Instructions & Tools** is a core building block in 06-google-gemini-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around System Instructions & Tools typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For System Instructions & Tools: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain System Instructions & Tools as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating System Instructions & Tools as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for System Instructions & Tools
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use System Instructions & Tools when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "System Instructions & Tools" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "System Instructions & Tools"
    notebook: str = "06-google-gemini-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — System Instructions & Tools

**Situation:** A team wants to productionize a feature involving **System Instructions & Tools**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## AI Studio vs Vertex

### Definition
**AI Studio vs Vertex** helps you choose among alternatives using explicit criteria rather than hype.

### Why it matters
In LLM API integration, weak designs around AI Studio vs Vertex typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
List options, define criteria (quality, cost, latency, ops, lock-in), score with evidence, document the decision.

### Intuition
Explain AI Studio vs Vertex as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating AI Studio vs Vertex as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for AI Studio vs Vertex
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use AI Studio vs Vertex when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "AI Studio vs Vertex" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "AI Studio vs Vertex"
    notebook: str = "06-google-gemini-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "AI Studio vs Vertex"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "AI Studio vs Vertex"}
strong = {"definition": "AI Studio vs Vertex", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "AI Studio vs Vertex"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "AI Studio vs Vertex", "passed": len(checks)-len(failed), "failed": failed})


## Comparison Snapshot

Use this table when reviewing designs in **Google Gemini API**.

| Topic | Do | Don't |
|-------|----|-------|
| Overview | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Text Generation | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Multimodal | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Embeddings | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Safety Settings | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| System Instructions & Tools | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Overview | Key concept covered in this notebook; see its section for definition and pitfalls |
| Text Generation | Key concept covered in this notebook; see its section for definition and pitfalls |
| Multimodal | Key concept covered in this notebook; see its section for definition and pitfalls |
| Embeddings | Key concept covered in this notebook; see its section for definition and pitfalls |
| Safety Settings | Key concept covered in this notebook; see its section for definition and pitfalls |
| System Instructions & Tools | Key concept covered in this notebook; see its section for definition and pitfalls |
| AI Studio vs Vertex | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Google Gemini API** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **08-llm-apis**.


## Try It Yourself

1. Implement a failing test/fixture for **Overview**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Text Generation**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Multimodal**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Embeddings**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Safety Settings**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
